In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('RetailCleaned.csv', header=None)

# Assigning headers
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate','Price', 'Customer_ID', 'Country', 'SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])   # Type conversion for date
print(df.shape)
df.info()

C:\Users\91935\AppData\Local\Temp\ipykernel_16412\2719874905.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('RetailCleaned.csv', header=None)


(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
 8   SourceSheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [8]:
# Filter and Calculate Recency, frequency, Mometary(given periods of time how much they purcase)

rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()          # create a copy of a data
snapshot_date = pd.to_datetime('2011-12-10')   # last date in data set is 9 dec 2011 so we take next day 10 dec

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),    # how recent he purcase clculated as 
    Frequency = ('Invoice', 'nunique'),                                     # it give number how much they buy
    Monetary = ('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())          # check quantiy * price and last take sum of that
).reset_index()


In [9]:
# Score customers and assign bussiness segments
# mosty given from client

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] >= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Need Attension'
    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [11]:
# Merge RFM results back onto transactions and export

df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()         # Creating new Data fame for juniun customers
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()   # It addonly month from invoice date
df['Revenue'] = df['Quantity'] * df['Price']                                     # total revenue

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency','Monetary','R_Score', 'F_Score', 'M_Score', 'Segment','IsChurned']],
    on = 'Customer_ID', how = 'left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index = False)

# index = false means (python as created default index like from 0- to n ) that index we don't want
# we want our custom index define above that we want 